In [1]:
import os
from dotenv import load_dotenv

print("Exists:", os.path.exists("../openai.env"))

load_dotenv("../openai.env", override=True)
print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

Exists: True
Key loaded: True


In [2]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

/tmp/ipykernel_91/1505468920.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader


In [3]:
load_dotenv("../openai.env")

True

# 1. Loading Documents


In [4]:
import requests

topics = ["Photosynthesis", "Water cycle", "Cellular respiration",
          "Plate tectonics", "Food chain"]

out_dir = "../resources"          # notebooks/ -> SemProject_MLEngine/resources
os.makedirs(out_dir, exist_ok=True)

headers = {"User-Agent": "SemProject-MLEngine/1.0 (student project)"}

for title in topics:
    r = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "prop": "extracts", "explaintext": 1,
                "titles": title, "redirects": 1, "format": "json"},
        headers=headers, timeout=30,
    )
    page = next(iter(r.json()["query"]["pages"].values()))
    words = page["extract"].split(" ")
    text = " ".join(words[:1500])

    fname = f"{out_dir}/{title.lower().replace(' ', '_')}.txt"
    with open(fname, "w", encoding="utf-8") as f:
        f.write(f"Source: Wikipedia, '{title}' (CC BY-SA)\n\n{text}")
    print(f"Saved {fname} ({len(text.split())} words)")

Saved ../resources/photosynthesis.txt (1525 words)
Saved ../resources/water_cycle.txt (1524 words)
Saved ../resources/cellular_respiration.txt (1527 words)
Saved ../resources/plate_tectonics.txt (1520 words)
Saved ../resources/food_chain.txt (1309 words)


In [5]:
docs_path ="../resources"
loader= DirectoryLoader(docs_path, glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding":"utf-8"},)

documents=loader.load()
print(f"Loaded {len(documents)} documents")

Loaded 5 documents


## 2. Splitting Documents into Chunks

In [6]:
splitter= CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks= splitter.split_documents(documents)
print (f"Created {len(chunks)} chunks")

Created a chunk of size 1849, which is longer than the specified 1000
Created a chunk of size 1024, which is longer than the specified 1000
Created a chunk of size 1628, which is longer than the specified 1000
Created a chunk of size 1273, which is longer than the specified 1000
Created a chunk of size 2277, which is longer than the specified 1000
Created a chunk of size 2290, which is longer than the specified 1000
Created a chunk of size 4085, which is longer than the specified 1000
Created a chunk of size 2058, which is longer than the specified 1000
Created a chunk of size 1191, which is longer than the specified 1000
Created a chunk of size 2840, which is longer than the specified 1000
Created a chunk of size 3911, which is longer than the specified 1000
Created a chunk of size 1720, which is longer than the specified 1000
Created a chunk of size 2630, which is longer than the specified 1000
Created a chunk of size 2841, which is longer than the specified 1000
Created a chunk of s

Created 42 chunks


## 3. Embedding and Storing

In [7]:
embeddings= OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore= Chroma.from_documents(documents=chunks,
                                   embedding=embeddings,
                                   persist_directory="db/chroma_db",)
print("Vector store created")

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}